In [11]:
import rohub
import uuid
import json
import datetime
import numpy as np
from pathlib import Path

## Extract metadata from the Zarr dataset

Read the `zarr.json` consolidated metadata to populate the RO-Crate automatically.

In [12]:
import fsspec
import zarr

zarr_url = "https://pangeo-eosc-minioapi.vm.fedcloud.eu/afouilloux-riomar/small_hp_pyramid.zarr"

# Read root metadata from remote Zarr store
mapper = fsspec.get_mapper(zarr_url)
root_meta = json.loads(mapper["zarr.json"])
root_attrs = root_meta["attributes"]

# Read level-0 consolidated metadata for variable attributes
level0_meta = json.loads(mapper["0/zarr.json"])
var_meta = level0_meta["consolidated_metadata"]["metadata"]

# Extract dataset info
dataset_title = root_attrs["title"]
dataset_name = root_attrs["name"]
dggs_info = root_attrs["dggs"]

print(f"Dataset: {dataset_name} ({dataset_title})")
print(f"Grid: HEALPix level {dggs_info['refinement_level']}, "
      f"nside={2**dggs_info['refinement_level']}, "
      f"indexing={dggs_info['indexing_scheme']}")

# Extract data variables only (exclude coordinates and grid variables)
coordinate_vars = {"cell_ids", "crs", "s_rho", "time_counter", "time_instant"}
data_vars = {k: v for k, v in var_meta.items()
             if k not in coordinate_vars
             and "attributes" in v and "long_name" in v.get("attributes", {})}
for vname, vmeta in data_vars.items():
    attrs = vmeta["attributes"]
    print(f"  {vname}: {attrs['long_name']} ({attrs.get('units', 'N/A')})")

# Extract temporal coverage from time_counter attributes
time_attrs = var_meta["time_counter"]["attributes"]
time_origin = datetime.datetime.strptime(time_attrs["time_origin"], "%Y-%m-%d %H:%M:%S")
n_timesteps = var_meta["time_counter"]["shape"][0]
print(f"\nTime origin: {time_origin}, {n_timesteps} hourly timesteps")

# Vertical levels
n_levels = var_meta["s_rho"]["shape"][0]
print(f"Vertical S-coordinate levels (s_rho): {n_levels}, bottom at level 0")

Dataset: GAMAR_GLORYS_1h_inst (GAMAR_GLORYS)
Grid: HEALPix level 13, nside=8192, indexing=nested
  salt: salinity (PSU)
  temp: potential temperature (Celsius)
  zeta: free-surface (meter)

Time origin: 2000-01-01 00:00:00, 2208 hourly timesteps
Vertical S-coordinate levels (s_rho): 40, bottom at level 0


In [13]:
# Compute actual temporal range from time data (seconds since time_origin)
store = zarr.open(fsspec.get_mapper(zarr_url + "/0"), mode="r")
time_data = store["time_counter"][:]
t_start = time_origin + datetime.timedelta(seconds=float(time_data.min()))
t_end = time_origin + datetime.timedelta(seconds=float(time_data.max()))
print(f"Temporal coverage: {t_start.date()} / {t_end.date()}")

temporal_coverage = f"{t_start.strftime('%Y-%m-%d')}/{t_end.strftime('%Y-%m-%d')}"

Temporal coverage: 2023-10-01 / 2023-12-31


## Set FAIR2Adapt RO-Crate profile terms

In [14]:
related_product = "https://schema.org/isRelatedTo"
has_type = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
Product = "https://schema.org/Product"
variableMeasured = "https://schema.org/variableMeasured"
Variable = "https://w3id.org/iadopt/ont/Variable"
name = "https://schema.org/name"
url = "https://schema.org/url"
depth = "https://schema.org/depth"
value = "https://schema.org/value"
temporalCoverage = "https://schema.org/temporalCoverage"
spatialCoverage = "https://schema.org/spatialCoverage"
Place = "https://schema.org/Place"
scenarioId = "https://w3id.org/ro/terms/cca/scenarioId"
spatialResolutionAsText = "http://data.europa.eu/930/spatialResolutionAsText"
license = "https://schema.org/license"
identifier = "https://schema.org/identifier"
provenance = "http://purl.org/dc/terms/provenance"

## Login to ROHub and load existing RO

In [15]:
credentials = json.loads(Path("~/rohub_credentials.json").expanduser().read_text())
rohub.login(username=credentials["username"], password=credentials["password"])

Logged successfully as annef@simula.no.


In [16]:
ro_pid = "https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056"
ro_id = ro_pid.rstrip("/").split("/")[-1]
annotation_id = ""  # leave empty to create a new annotation

ro = rohub.ros_load(identifier=ro_id)

Research Object was successfully loaded with id = 24600867-23ca-4b97-be14-3aa63883c056


## Create annotation and set resource identifiers

In [19]:
if not annotation_id:
    new_annot = ro.add_annotations()
    annotation_id = new_annot["identifier"]
print("annotation_id:", annotation_id)

annotation_id: 381c5725-36fd-4418-a8e1-655cab3f4738


In [20]:
# Product (MVP)
mvp_id = f"https://w3id.org/ro-id/{ro_id}/product/1"

# Variables — one per data variable in the Zarr store
# Map Zarr variable names to CF standard name NERC vocab URIs
cf_standard_names = {
    "salt": {
        "name": "sea_water_salinity",
        "url": "http://vocab.nerc.ac.uk/standard_name/sea_water_salinity/",
    },
    "temp": {
        "name": "sea_water_potential_temperature",
        "url": "http://vocab.nerc.ac.uk/standard_name/sea_water_potential_temperature/",
    },
    "zeta": {
        "name": "sea_surface_height_above_geoid",
        "url": "http://vocab.nerc.ac.uk/standard_name/sea_surface_height_above_geoid/",
    },
}

variable_ids = {}
for i, vname in enumerate(data_vars, start=1):
    variable_ids[vname] = f"https://w3id.org/ro-id/{ro_id}/variable/{i}"

# Spatial coverage
spatial_id = f"https://w3id.org/ro-id/{ro_id}/spatial/1"

# Depth value
depth_value_id = f"https://w3id.org/ro-id/{ro_id}/value/{uuid.uuid4()}"

print(f"Product: {mvp_id}")
print(f"Variables: {variable_ids}")
print(f"Spatial: {spatial_id}")

Product: https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/product/1
Variables: {'salt': 'https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/variable/1', 'temp': 'https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/variable/2', 'zeta': 'https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/variable/3'}
Spatial: https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/spatial/1


## Set values from dataset metadata

In [21]:
# Depth: S-coordinate vertical levels, bottom at level 0
depth_value_text = f"S-coordinate (s_rho), {n_levels} vertical levels, bottom at level 0"

# Temporal coverage: extracted from data
temporalCoverage_value = temporal_coverage

# Scenario: from dataset title (model forcing configuration)
scenarioId_value = dataset_name  # GAMAR_GLORYS_1h_inst

# Spatial resolution: HEALPix level from dggs metadata
hp_level = dggs_info["refinement_level"]
spatialResolutionAsText_value = f"HEALPix level {hp_level} (nside={2**hp_level})"

# Licenses
license_data_value = "https://creativecommons.org/licenses/by/4.0/"
license_code_value = "https://opensource.org/licenses/MIT"

# Dataset identifier (Zarr URL)
mvp_id_value = zarr_url

# Provenance: link to the dashboard
provenance_url_value = "https://fair2adapt.github.io/riomar-dashboard/"

# Spatial coverage
spatialCoverage_name_value = "Bay of Biscay"
spatialCoverage_url_value = "https://www.geonames.org/2960858"

print(f"Depth: {depth_value_text}")
print(f"Temporal: {temporalCoverage_value}")
print(f"Scenario: {scenarioId_value}")
print(f"Resolution: {spatialResolutionAsText_value}")
print(f"License (data): {license_data_value}")
print(f"License (code): {license_code_value}")
print(f"Spatial: {spatialCoverage_name_value}")

Depth: S-coordinate (s_rho), 40 vertical levels, bottom at level 0
Temporal: 2023-10-01/2023-12-31
Scenario: GAMAR_GLORYS_1h_inst
Resolution: HEALPix level 13 (nside=8192)
License (data): https://creativecommons.org/licenses/by/4.0/
License (code): https://opensource.org/licenses/MIT
Spatial: Bay of Biscay


## Create all annotations on the RO-Crate

In [22]:
# --- Connect product to RO-Crate ---
ro.add_triple(
    the_subject=ro_pid, the_predicate=related_product,
    the_object=mvp_id, annotation_id=annotation_id, object_class="URIRef",
)

# Set product type
ro.add_triple(
    the_subject=mvp_id, the_predicate=has_type,
    the_object=Product, annotation_id=annotation_id, object_class="URIRef",
)

# --- Variables (from Zarr metadata + CF standard names) ---
for vname, vid in variable_ids.items():
    zarr_attrs = data_vars[vname]["attributes"]
    cf = cf_standard_names[vname]

    ro.add_triple(
        the_subject=mvp_id, the_predicate=variableMeasured,
        the_object=vid, annotation_id=annotation_id, object_class="URIRef",
    )
    ro.add_triple(
        the_subject=vid, the_predicate=has_type,
        the_object=Variable, annotation_id=annotation_id, object_class="URIRef",
    )
    # Use long_name from Zarr + CF standard name
    var_label = f"{zarr_attrs['long_name']} ({cf['name']})"
    ro.add_triple(
        the_subject=vid, the_predicate=name,
        the_object=var_label, annotation_id=annotation_id,
    )
    ro.add_triple(
        the_subject=vid, the_predicate=url,
        the_object=cf["url"], annotation_id=annotation_id, object_class="URIRef",
    )

# --- Depth (vertical coordinate) ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=depth,
    the_object=depth_value_id, annotation_id=annotation_id, object_class="URIRef",
)
ro.add_triple(
    the_subject=depth_value_id, the_predicate=value,
    the_object=depth_value_text, annotation_id=annotation_id,
)

# --- Temporal coverage ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=temporalCoverage,
    the_object=temporalCoverage_value, annotation_id=annotation_id,
)

# --- Spatial coverage (Bay of Biscay) ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=spatialCoverage,
    the_object=spatial_id, annotation_id=annotation_id, object_class="URIRef",
)
ro.add_triple(
    the_subject=spatial_id, the_predicate=has_type,
    the_object=Place, annotation_id=annotation_id, object_class="URIRef",
)
ro.add_triple(
    the_subject=spatial_id, the_predicate=name,
    the_object=spatialCoverage_name_value, annotation_id=annotation_id,
)
ro.add_triple(
    the_subject=spatial_id, the_predicate=url,
    the_object=spatialCoverage_url_value, annotation_id=annotation_id, object_class="URIRef",
)

# --- Scenario ID ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=scenarioId,
    the_object=scenarioId_value, annotation_id=annotation_id,
)

# --- Spatial resolution ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=spatialResolutionAsText,
    the_object=spatialResolutionAsText_value, annotation_id=annotation_id,
)

# --- License (CC-BY-4.0 for data) ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=license,
    the_object=license_data_value, annotation_id=annotation_id, object_class="URIRef",
)

# --- Dataset identifier (Zarr URL) ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=identifier,
    the_object=mvp_id_value, annotation_id=annotation_id,
)

# --- Provenance (dashboard URL) ---
ro.add_triple(
    the_subject=mvp_id, the_predicate=provenance,
    the_object=provenance_url_value, annotation_id=annotation_id, object_class="URIRef",
)

print("All annotations created successfully.")

All annotations created successfully.


In [18]:
# Delete annotation (if needed — uncomment and set the annotation ID)
# annot_id = "paste-annotation-id-here"
#annot_id = "123480b8-957b-4b62-9afc-e2b3257d42cb"
#rohub.annotation_delete(annotation_identifier=annot_id)

Annotation successfully deleted!
